# Simplex method benchmarks

In [6]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import sys
# sys.path.append('.')
# sys.path.append('./DynaMix/')

plt.rcParams["font.family"] = "Helvetica"

# Lyapunov exponent calculation

In [8]:
import sys
## add the path to the project
sys.path.append('..')

from models.parrot import SimplexForecaster, context_parroting_forecast


In [43]:


# ?max_lyapunov_exponent_rosenstein
import sys
## add the path to the project
sys.path.append('..')

from models.parrot import SimplexForecaster


In [65]:
import torch
from chronos import BaseChronosPipeline, ChronosPipeline, ChronosBoltPipeline

pipeline = BaseChronosPipeline.from_pretrained(
    "amazon/chronos-t5-base",  # use "amazon/chronos-bolt-small" for the corresponding Chronos-Bolt model
    #"amazon/chronos-bolt-base",
    #device_map="cuda",  # use "cpu" for CPU inference
    device_map="mps",
    torch_dtype=torch.bfloat16,
)

In [34]:
import sys
# sys.path.append('.')
sys.path.append('./DynaMix/')

import torch
# from src.model.dynamix import DynaMix
from src.model.forecaster import DynaMixForecaster
# from src.metrics.metrics import geometrical_misalignment, temporal_misalignment, MASE
# from src.utilities.plotting_eval import plot_3D_attractor, plot_2D_attractor, plot_TS_forecast
from src.utilities.utilities import load_hf_model

# Load the pre-trained model
model = load_hf_model("dynamix-3d-alrnn-v1.0")
model.eval() # Set model to evaluation mode
forecaster = DynaMixForecaster(model) # Initialize the forecaster

/Users/william/program_repos/parroting/benchmark/DynaMix/src/model/dynamix.py:95: RuntimeWarning: divide by zero encountered in matmul
  K = R.T @ R / M + np.eye(M)
/Users/william/program_repos/parroting/benchmark/DynaMix/src/model/dynamix.py:95: RuntimeWarning: overflow encountered in matmul
  K = R.T @ R / M + np.eye(M)
/Users/william/program_repos/parroting/benchmark/DynaMix/src/model/dynamix.py:95: RuntimeWarning: invalid value encountered in matmul
  K = R.T @ R / M + np.eye(M)


In [66]:
import glob
from dysts.analysis import max_lyapunov_exponent_rosenstein, max_lyapunov_exponent_rosenstein_multivariate, gp_dim
from dysts.metrics import estimate_kl_divergence


results_path = "../analysis/Chronos_statistics/"
CHECK_EXISTING = False


forecast_length = 300
context_lengths = np.unique(np.logspace(np.log10(forecast_length), np.log10(4000), 10).astype(int))



if CHECK_EXISTING:
    all_lyap = np.load(results_path + '/all_lyap_vary_context.npy', allow_pickle=True).tolist()
    all_cdim = np.load(results_path + '/all_cdim_vary_context.npy', allow_pickle=True).tolist()
    all_kl_dist = np.load(results_path + '/all_kl_dist_vary_context.npy', allow_pickle=True).tolist()
else:
    all_lyap = list()
    all_cdim = list()
    all_kl_dist = list()

for trajectory in sorted(glob.glob("../data/long_trajectories/*.npy")):
    equation_name = trajectory.split("/")[-1].split(".")[0]
    print(equation_name, flush=True)
    # if equation_name in found_systems:
    #     print(f"Skipping {equation_name} because it's already in the found_systems list", flush=True)
    #     continue
    
    traj = np.load(trajectory, allow_pickle=True)
    traj = (traj - np.mean(traj, axis=0)) / np.std(traj, axis=0)

    try:
        
        all_cdim_inner = []
        all_kl_dist_inner = []
        all_lyap_inner = []
        for context_length in context_lengths:
            traj_context = traj[:context_length, :]
            traj_true = traj[context_length:context_length+forecast_length, :]



            # traj_pred = list()
            # for mode in range(traj_context.shape[1]):
            #     traj_pred_mode = context_parroting_forecast(traj_context[:, mode], forecast_total_length=forecast_length)[2]
            #     traj_pred.append(traj_pred_mode)
            # traj_pred = np.array(traj_pred).T

            # traj_pred = list()
            # for mode in range(traj_context.shape[1]):
            #     model = SimplexForecaster()
            #     model.fit(traj_context[:, mode])
            #     traj_pred_mode = model.forecast(forecast_length)
            #     traj_pred.append(traj_pred_mode)
            # traj_pred = np.array(traj_pred).T


            traj_pred = list()
            for mode in range(traj_context.shape[1]):
                forecast = pipeline.predict(
                    inputs=torch.tensor(traj_context[:, mode]),
                    prediction_length=forecast_length,
                    num_samples=20,
                    limit_prediction_length=False,
                    )
                traj_pred_mode = np.mean(forecast[0, :, :].detach().numpy(), axis=0)
                traj_pred.append(traj_pred_mode)
            traj_pred = np.array(traj_pred).T


            # context_traj_tensor  = torch.tensor(traj_context)
            # with torch.no_grad(): 
            #     reconstruction = forecaster.forecast(
            #         context=context_traj_tensor,
            #         horizon=forecast_length, # Match the horizon of the ground truth
            #         standardize=True,
            #     )
            # traj_pred = reconstruction.detach().numpy()

            
            kl_dist = estimate_kl_divergence(traj_true, traj_pred)
            if np.isinf(kl_dist): kl_dist = np.nan
            
            
            cdim_true = gp_dim(traj_true)
            cdim_pred = gp_dim(traj_pred)
            cdim = np.array([cdim_pred, cdim_true])

            lyap_true = max_lyapunov_exponent_rosenstein(traj_true)
            if np.isinf(lyap_true): lyap_true = 0
            lyap_pred = max_lyapunov_exponent_rosenstein(traj_pred)
            if np.isinf(lyap_pred): lyap_pred = 0
            lyap = np.array([lyap_pred, lyap_true])


            all_cdim_inner.append(cdim)
            all_kl_dist_inner.append(kl_dist)
            all_lyap_inner.append(lyap)

        all_cdim.append(all_cdim_inner)
        all_kl_dist.append(all_kl_dist_inner)
        all_lyap.append(all_lyap_inner)

        # np.array(all_cdim).dump(results_path + '/all_cdim_vary_context.npy')
        # np.array(all_kl_dist).dump(results_path + '/all_kl_dist_vary_context.npy')
        # np.array(all_lyap).dump(results_path + '/all_lyap_vary_context.npy')


    except Exception as e:
        print(e)
        print(f"Skipping {equation_name}", flush=True)
        continue


Aizawa


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
/Users/william/program_repos/parroting/.venv/lib/python3.13/site-packages/scipy/stats/_covariance.py:633: RuntimeWarning: divide by zero encountered in matmul
  return x @ self._LP
/Users/william/program_repos/parroting/.venv/lib/python3.13/site-packages/scipy/stats/_covariance.py:633: RuntimeWarning: overflow encountered in matmul
  return x @ self._LP
/Users/william/program_repos/parroting/.venv/lib/python3.13/site-packages/scipy/stats/_covariance.py:633: RuntimeWarning: invalid value encountered in matmul
  return x @ self._LP
We recommend keeping prediction length <= 64. The quality of l

AnishchenkoAstakhov


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

Arneodo


...

We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

YuWang2


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

ZhouChen


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

In [67]:
results_path = "../analysis/Chronos_statistics/"
np.array(all_cdim).dump(results_path + '/all_cdim_vary_context.npy')
np.array(all_kl_dist).dump(results_path + '/all_kl_dist_vary_context.npy')
np.array(all_lyap).dump(results_path + '/all_lyap_vary_context.npy')

In [ ]:
from scipy.stats import pearsonr, spearmanr

model_names = ["parrot", "dynamix", "simplex", "Chronos"]
# model_names = ["parrot", "dynamix",  "simplex"]

import degas as dg
colors = [dg.red, dg.blue, dg.turquoise, dg.orange][:len(model_names)]


context_lengths = np.unique(np.logspace(np.log10(forecast_length), np.log10(4000), 10).astype(int))

from scipy.stats import norm
def err_from_corr(corr, pvalue):
    z_obs = np.arctanh(corr)
    z_stat = abs(norm.ppf(pvalue / 2))
    sd_z = abs(z_obs) / z_stat if z_stat != 0 else np.nan
    sd_r = sd_z * (1 - corr**2)
    return sd_r

plt.figure(figsize=(15, 4.8))

plt.subplot(1, 3, 1)
for model_name, color in zip(model_names, colors):
    results_path = f"../analysis/{model_name}_statistics/"
    all_cdim = np.load(results_path + '/all_cdim_vary_context.npy', allow_pickle=True)

    all_cdim = np.array(all_cdim)
    all_cdim_transposed = all_cdim.transpose(1, 0, 2)
    
    corrs = [spearmanr(cdim_pair[:, 0], cdim_pair[:, 1])[0] for cdim_pair in all_cdim_transposed]
    corr_errs = [err_from_corr(corr, spearmanr(cdim_pair[:, 0], cdim_pair[:, 1])[1]) for corr, cdim_pair in zip(corrs, all_cdim_transposed)]
    # plt.semilogx(
    #     context_lengths, [spearmanr(lyap_pair[:, 0], lyap_pair[:, 1])[0] for lyap_pair in all_lyap_transposed], label=model_name.capitalize()
    # )
    corrs = [spearmanr(cdim_pair[:, 0], cdim_pair[:, 1])[0] for cdim_pair in all_cdim_transposed]
    corr_errs = [err_from_corr(corr, spearmanr(cdim_pair[:, 0], cdim_pair[:, 1])[1]) for corr, cdim_pair in zip(corrs, all_cdim_transposed)]
    plt.semilogx()
    dg.plot_err(np.array(corrs), np.array(corr_errs), x=context_lengths, label=model_name.capitalize(), color=color, linewidth=2)

plt.xlim(context_lengths[0], context_lengths[-1])
plt.legend(frameon=False)
plt.xlabel("Context length")
plt.ylabel("Correlation")


plt.subplot(1, 3, 2)
for model_name, color in zip(model_names, colors):
    results_path = f"../analysis/{model_name}_statistics/"
    all_lyap = np.load(results_path + '/all_lyap_vary_context.npy', allow_pickle=True)

    all_lyap = np.array(all_lyap)
    all_lyap_transposed = all_lyap.transpose(1, 0, 2)
    ## Capitalize the first letter of the model name
    corrs = [spearmanr(lyap_pair[:, 0], lyap_pair[:, 1])[0] for lyap_pair in all_lyap_transposed]
    corr_errs = [err_from_corr(corr, spearmanr(lyap_pair[:, 0], lyap_pair[:, 1])[1]) for corr, lyap_pair in zip(corrs, all_lyap_transposed)]
    # plt.semilogx(
    #     context_lengths, [spearmanr(lyap_pair[:, 0], lyap_pair[:, 1])[0] for lyap_pair in all_lyap_transposed], label=model_name.capitalize()
    # )
    plt.semilogx()
    dg.plot_err(np.array(corrs), np.array(corr_errs), x=context_lengths, label=model_name.capitalize(), color=color, linewidth=2)

plt.xlim(context_lengths[0], context_lengths[-1])
plt.legend(frameon=False)
plt.xlabel("Context length")
plt.ylabel("Correlation")


plt.subplot(1, 3, 3)
for model_name, color in zip(model_names, colors):
    results_path = f"../analysis/{model_name}_statistics/"
    all_kl_dist = np.load(results_path + '/all_kl_dist_vary_context.npy', allow_pickle=True)

    all_kl_dist = np.array(all_kl_dist)

    ## Capitalize the first letter of the model name
    # plt.semilogx(
    #     context_lengths, np.mean(all_kl_dist, axis=0), label=model_name.capitalize()
    # )
    plt.semilogx()
    dg.plot_err(
        np.mean(all_kl_dist, axis=0), np.std(all_kl_dist, axis=0) / np.sqrt(len(all_kl_dist)),
         x=context_lengths, label=model_name.capitalize(), color=color, linewidth=2
    )
plt.xlim(context_lengths[0], context_lengths[-1])
plt.legend(frameon=False)
plt.xlabel("Context length")
plt.ylabel("KL Divergence")

    # all_lyap = np.array(all_lyap)
    # all_lyap_transposed = all_lyap.transpose(1, 0, 2)
    # plt.plot(context_lengths, [spearmanr(lyap_pair[:, 0], lyap_pair[:, 1])[0] for lyap_pair in all_lyap_transposed], label=model_name)

    # all_kl_dist = np.array(all_kl_dist)
    # plt.figure()
    # plt.plot(context_lengths, np.mean(all_kl_dist, axis=0))

    # print(model_name)
    # print(f"KL: {np.mean(all_kl_dist)}, {np.std(all_kl_dist) / np.sqrt(len(all_kl_dist))}")
    # print(spearmanr(np.array(all_cdim)[:,0], np.array(all_cdim)[:,1])[0], err_from_corr(spearmanr(np.array(all_cdim)[:,0], np.array(all_cdim)[:,1])[0], spearmanr(np.array(all_cdim)[:,0], np.array(all_cdim)[:,1])[1]))
    # print(spearmanr(np.array(all_lyap)[:,0], np.array(all_lyap)[:,1])[0], err_from_corr(spearmanr(np.array(all_lyap)[:,0], np.array(all_lyap)[:,1])[0], spearmanr(np.array(all_lyap)[:,0], np.array(all_lyap)[:,1])[1]))
    # # print(spearmanr(np.array(all_cdim)[:,0], np.array(all_cdim)[:,1]))
    # # print(spearmanr(np.array(all_lyap)[:, 0], np.array(all_lyap)[:, 1]))
    # print("\n")

dg.better_savefig("../analysis/vary_context.pdf", dpi=300, transparent=True)
